# Muon Decay Anaysis

For: **XXI Mexican School on Particles and Fields 2026**

Instructor: **Haydee Hernandez**

Cosmic-ray interactions provide muons that can reach a surface detector. A stopped muon can subsequently decay, and the time between the stopping signal and decay signal carries information about its lifetime.

For the dominant weak decay modes we can write

$$\mu^-\to e^-+\bar\nu_e+\nu_\mu,$$
$$\mu^+\to e^++\nu_e+\bar\nu_\mu.$$

If every muon has the same lifetime parameter, why do individual muons decay at different measured times?

## Extracting Real Muon Decay Data from CIIEC Database

Download the CSV dataset from the [CIIEC experiment page](https://ciiec.buap.mx/Muon-Decay-Real-Data/) and obtain the decay time.

**NOTICE** how the data looks. How would you extract the time between two pulses from the data?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import os

In [ ]:
#EXTRACTING DELTA TIME


# Data File
fileDeltas = 'MD20231005_1151.csv'
if not os.path.isfile(fileDeltas):
    print("File not found. Downloading...")
    !wget "https://ciiec.buap.mx/Muon-Decay-Real-Data/MD20231005_1151.csv" --no-check-certificate
# Delta_t outputs
outputfile = 'DeltaDecay_2023.csv'

import csv
import re

def extract_delta_time_from_field(field):
    """Extract delta time from a CSV field.
    
    Notice how the file includes the calculation of the time between the first and second pulse.
    
    We can extract it by using the function "search". """
    pattern = r'Measured Delta Time Between first and second trigger \(ns\)\s*=\s*([-\d.]+)'
    match = re.search(pattern, field)
    return match.group(1) if match else None

def process_csv_alternative(input_file, output_file, max_lines=2000):
    """Read the CSV file, find the Delta Time and store it into the outputfile."""
    delta_times = []
    
    try:
        with open(input_file, 'r', encoding='utf-8') as file:
                    csv_reader = csv.reader(file)
                    
                    for i, row in enumerate(csv_reader):
                        if i >= max_lines:
                            break
                        
                        # Assuming your data is in a single column (first column)
                        if row:  # Check if row is not empty
                            delta_time = extract_delta_time_from_field(row[0])
                            delta_times.append([delta_time] if delta_time else [''])
                        else:
                            delta_times.append([''])
        
    
    except Exception as e:
        print(f"Error: {e}")
        return
    
    # Write to output file
    try:
        with open(output_file, 'w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(['Delta_Time_ns'])
            writer.writerows(delta_times)
        
        print(f"Processed {len(delta_times)} lines to '{output_file}'")
        
    except Exception as e:
        print(f"Error writing file: {e}")

# If the       
process_csv_alternative(fileDeltas, outputfile, max_lines=100000)

### Using PANDAS to analyze data

In [ ]:
outputfile = 'DeltaDecay_2023.csv'

df = pd.read_csv(outputfile)
display(df.head())
print("Shape:", df.shape)
print("Columns:", list(df.columns))
print(df.dtypes)

Another useful command:

In [ ]:
df.info()

## Plotting histogram

In [ ]:
##  STYLING WITH SEABORN

import seaborn as sns

custom_rc = {
    "font.family": "serif",
    "axes.titlesize": 18,
    "axes.titleweight": "bold",
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.labelcolor": "black",
    "xtick.labelcolor": "black",
    "ytick.labelcolor": "black",
    "axes.edgecolor": "black", 
    "axes.linewidth": 2.0,  
}

sns.set_theme("paper",rc=custom_rc,palette='colorblind')
sns.set_style("whitegrid", {"axes.facecolor": ".95"}) # adding gridlines

In [ ]:
def plot_histogram(df,nbins):
    plt.figure(figsize=(8,5))
    plt.hist(df, bins=nbins, histtype='bar')
    plt.title(f"{nbins} bins")
    plt.xlabel(r"$\Delta t$ [ns]")
    plt.ylabel("Events")
    plt.tight_layout()
    
plot_histogram(df,100)

Try different number of bins.

*What can you conclude?*

In [ ]:
# EXERCISE 1 - YOUR CODE HERE



## Exercise 2 - Choosing a Window for Fitting

Choose a fit window after looking at the data. Do not silently tune it until the fitted lifetime looks correct.

A useful histogram representation is the bin center $t_i$, observed count $n_i$, and approximate uncertainty $\sqrt{n_i}$.

In [ ]:
# EXERCISE 2 — Choose these from the observed dataset
T_MIN = 0
T_MAX = 1000.0      # us; CHANGE if the actual data require another range
N_BINS = 20

def plot_errors(df,t_min=T_MIN,t_max=T_MAX,nbins=N_BINS):
    fit_sample = df[(df.Delta_Time_ns >= t_min) & (df.Delta_Time_ns <= t_max)]
    counts, edges = np.histogram(fit_sample, bins=nbins, range=(t_min, t_max))
    centers = 0.5 * (edges[:-1] + edges[1:])
    errors = np.sqrt(counts)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(centers, counts, yerr=errors, fmt='o', ms=4, capsize=2)
    ax.set_xlabel(r"$\Delta t$ [ns]")
    ax.set_ylabel("Events / bin")
    ax.set_title("Muon decay-time candidates")
    plt.show()
    
plot_errors(df,T_MIN,T_MAX,N_BINS)

## Exercise 3 — linear vs logarithmic scale
Plot the same histogram on a logarithmic vertical scale. What should a pure exponential function look like in this scale? What would a flat background look like?

In [ ]:
# EXERCISE 3 — Choose these from the observed dataset
T_MIN = 0
T_MAX = 1000.0      # us; CHANGE if the actual data require another range
N_BINS = 50
    
def logplot_errors(df,t_min=T_MIN,t_max=T_MAX,nbins=N_BINS):
    fit_sample = df[(df.Delta_Time_ns >= t_min) & (df.Delta_Time_ns <= t_max)]
    counts, edges = np.histogram(fit_sample, bins=nbins, range=(t_min, t_max))
    centers = 0.5 * (edges[:-1] + edges[1:])
    errors = np.sqrt(counts)

    plt.figure(figsize=(8, 5))
    plt.errorbar(centers[counts>0], counts[counts>0], yerr=errors[counts>0], fmt='o', ms=4, capsize=2)
    plt.yscale('log')
    plt.xlabel(r"$\Delta t$ [$n$s]")
    plt.ylabel("Events / bin")
    plt.title("Muon decay-time candidates")
    plt.show()
    
logplot_errors(df,T_MIN,T_MAX,N_BINS)

## 4. Fit an exponential plus background

We now fit

$$N(t)=A e^{-t/\tau}+B.$$

The direct nonlinear fit avoids taking $\ln(0)$ and keeps the physics model explicit.

For this introductory exercise we use `scipy.optimize.curve_fit` with approximate $\sqrt{n}$ histogram uncertainties. Empty bins require care because that approximation gives zero uncertainty; here we use only bins with positive counts. In an advanced analysis, a Poisson likelihood is preferable, especially for sparse bins.

In [ ]:
def decay_model(t, A, tau, B):
    return A * np.exp(-t / tau) + B

# EXERCISE 4 — Fit the data to an Exponential
T_MIN = 0
T_MAX = 1000.0      # Use the parameters that you found more suitable from previous exercises
N_BINS = 50

def fit_exp(df,t_min=T_MIN,t_max=T_MAX,nbins=N_BINS):
    fit_sample = df[(df.Delta_Time_ns >= t_min) & (df.Delta_Time_ns <= t_max)]
    counts, edges = np.histogram(fit_sample, bins=nbins, range=(t_min, t_max))
    centers = 0.5 * (edges[:-1] + edges[1:])
    errors = np.sqrt(counts)
    use = counts > 0
    t_fit = centers[use]
    n_fit = counts[use]
    sigma_fit = np.sqrt(n_fit)

    # Provide physically reasonable initial values from the displayed distribution.
    p0 = [max(n_fit) - min(n_fit), 2.2, max(0.1, np.median(n_fit[-5:]))]

    popt, pcov = curve_fit(
        decay_model, t_fit, n_fit,
        p0=p0,
        sigma=sigma_fit,
        absolute_sigma=True,
        bounds=([0, 0, 0], [np.inf, np.inf, np.inf]),
        maxfev=20000
    )

    A_hat, tau_hat, B_hat = popt
    perr = np.sqrt(np.diag(pcov))
    A_err, tau_err, B_err = perr

    print(f"A   = {A_hat:.4g} +/- {A_err:.3g} events/bin")
    print(f"tau = {tau_hat:.6g} +/- {tau_err:.3g} ns")
    print(f"B   = {B_hat:.4g} +/- {B_err:.3g} events/bin")
    
    
    
    tt = np.linspace(T_MIN, T_MAX, 500)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.errorbar(centers, counts, yerr=errors, fmt='o', ms=4, capsize=2, label='Data')
    ax.plot(tt, decay_model(tt, *popt), label='Exponential + background')
    ax.set_xlabel(r"$\Delta t$ [$\mu$s]")
    ax.set_ylabel("Events / bin")
    
    ax.legend()
    ax.set_title(rf"$\tau = {tau_hat:.3f} \pm {tau_err:.3f}\ \mu$s")
    plt.show()
    
    return np.array([[A_hat, A_err], [tau_hat, tau_err], [B_hat , B_err]])

results=fit_exp(df,T_MIN,T_MAX,N_BINS)


## 5. Fit quality: residuals and $\chi^2$

A parameter estimate is not enough. We also ask whether the fitted model describes the measurements.

Define normalized residuals

$$r_i=\frac{n_i-N_{\rm fit}(t_i)}{\sigma_i},$$

and

$$\chi^2=\sum_i r_i^2,\qquad
\chi^2_\nu=\frac{\chi^2}{N_{\rm dof}}.$$

For this model, three fitted parameters mean $N_{\rm dof}=N_{\rm used\ bins}-3$.

For this, add the following code to the function used before:

```python
expected = decay_model(t_fit, *popt)
residuals = (n_fit - expected) / sigma_fit
chi2 = np.sum(residuals**2)
ndof = len(n_fit) - len(popt)
chi2_red = chi2 / ndof
```

And to add it to the plot, add the line:

```python
ax.text(5,5,f"chi2 = {chi2:.2f},\nndof = {ndof}\nchi2/ndof = {chi2_red:.3f}")
```

## 6. Compare with the accepted free-muon lifetime

For the school comparison we use the reference supplied by the instructor. The value entered below is the one used in the accompanying course design; instructors should update it from the current Particle Data Group listing when preparing future editions.

$$\tau_{\rm ref}=2196.9811\,\mathrm{ns},\qquad
\sigma_{\rm ref}=0.0022\,\mathrm{ns}.$$

Calculate:

1. absolute difference $\Delta\tau=\tau_{\rm meas}-\tau_{\rm ref}$;
2. relative precision $100\,\sigma_\tau/\tau$;
3. Pull:

$$z=\frac{\tau_{\rm meas}-\tau_{\rm ref}}
{\sqrt{\sigma_{\rm meas}^2+\sigma_{\rm ref}^2}}.$$

A stopped-$\mu^-$ sample in matter can involve physics beyond free decay, so interpretation of a discrepancy requires understanding the material and charge composition as well as the statistics.

In [ ]:
# EXERCISE 6

# PDG NUMBERS
tau_ref = 2196.9811        # ns
tau_ref_err = 0.0022   # ns

tau_exp = results[1,0]
tau_exp_err = results[1,1]


delta_tau = ...
relative_precision = ...
pull = ...

print(f"Measured tau:           {tau_exp:.6f} +/- {tau_exp_err:.6f} ns")
print(f"Reference tau:          {tau_ref:.7f} +/- {tau_ref_err:.7f} ns")
print(f"Difference:             {delta_tau:+.6f} ns")
print(f"Relative stat. precision: {relative_precision:.2f}%")
print(f"Pull:                   {pull:+.2f} sigma")

What can you conclude from this exercise?